# TF-IDF Model Comparison

This notebook loads the shared TF-IDF features produced by `ml_tfidf_preprocessing.ipynb`
and trains all four classical models, reporting their validation scores.


In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from scipy.sparse import load_npz

from sklearn.svm import LinearSVC
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score, classification_report


In [2]:
RANDOM_STATE = 42
CLASS_WEIGHT = 'balanced'

ARTIFACTS_DIR = Path('../../artifacts/on_text/tfidf_baselines/v1')


In [3]:
X_train_vectors = load_npz(ARTIFACTS_DIR / 'X_train_vectors.npz')
X_valid_vectors = load_npz(ARTIFACTS_DIR / 'X_valid_vectors.npz')

y_train = np.load(ARTIFACTS_DIR / 'y_train.npy')
y_valid = np.load(ARTIFACTS_DIR / 'y_valid.npy')

label_names = json.loads((ARTIFACTS_DIR / 'label_names.json').read_text())
label_names = [str(label) for label in label_names]

print(f'Train matrix shape: {X_train_vectors.shape}')
print(f'Validation matrix shape: {X_valid_vectors.shape}')


Train matrix shape: (67932, 317321)
Validation matrix shape: (16984, 317321)


In [4]:
def evaluate_model(model_name, model, X_train, y_train, X_valid, y_valid):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_valid)

    metrics = {
        'model': model_name,
        'accuracy': accuracy_score(y_valid, y_pred),
        'f1_macro': f1_score(y_valid, y_pred, average='macro'),
        'f1_weighted': f1_score(y_valid, y_pred, average='weighted'),
    }

    report = classification_report(y_valid, y_pred, target_names=label_names)
    return metrics, report


In [5]:
results = []
reports = {}


## LinearSVC


In [6]:
model = LinearSVC(
    C=1.0,
    class_weight=CLASS_WEIGHT,
    random_state=RANDOM_STATE,
)
metrics, report = evaluate_model(
    'LinearSVC',
    model,
    X_train_vectors,
    y_train,
    X_valid_vectors,
    y_valid,
)
results.append(metrics)
reports['LinearSVC'] = report
print(metrics)
print(report)


{'model': 'LinearSVC', 'accuracy': 0.8532736693358455, 'f1_macro': 0.8389271481619359, 'f1_weighted': 0.8524857462808447}
              precision    recall  f1-score   support

          10       0.58      0.60      0.59       623
          40       0.76      0.73      0.75       502
          50       0.84      0.88      0.86       336
          60       0.90      0.85      0.88       166
        1140       0.78      0.82      0.80       534
        1160       0.94      0.96      0.95       791
        1180       0.74      0.67      0.70       153
        1280       0.79      0.71      0.75       974
        1281       0.65      0.62      0.64       414
        1300       0.93      0.97      0.95      1009
        1301       0.97      0.96      0.97       161
        1302       0.85      0.81      0.83       498
        1320       0.87      0.85      0.86       648
        1560       0.87      0.86      0.87      1015
        1920       0.91      0.93      0.92       861
        1940 

## SGDClassifier


In [7]:
model = SGDClassifier(
    loss='log_loss',
    alpha=1e-4,
    penalty='l2',
    max_iter=1000,
    random_state=RANDOM_STATE,
    n_iter_no_change=5,
    tol=1e-4,
    class_weight=CLASS_WEIGHT,
)
metrics, report = evaluate_model(
    'SGDClassifier',
    model,
    X_train_vectors,
    y_train,
    X_valid_vectors,
    y_valid,
)
results.append(metrics)
reports['SGDClassifier'] = report
print(metrics)
print(report)


{'model': 'SGDClassifier', 'accuracy': 0.7893900141309468, 'f1_macro': 0.7800109889292183, 'f1_weighted': 0.7918315507410884}
              precision    recall  f1-score   support

          10       0.36      0.70      0.48       623
          40       0.76      0.55      0.64       502
          50       0.76      0.84      0.80       336
          60       0.96      0.77      0.86       166
        1140       0.73      0.79      0.76       534
        1160       0.92      0.92      0.92       791
        1180       0.64      0.61      0.62       153
        1280       0.73      0.50      0.59       974
        1281       0.59      0.50      0.54       414
        1300       0.82      0.93      0.87      1009
        1301       0.95      0.96      0.95       161
        1302       0.78      0.72      0.75       498
        1320       0.80      0.74      0.77       648
        1560       0.82      0.77      0.79      1015
        1920       0.89      0.90      0.89       861
        1

## LogisticRegression


In [8]:
model = LogisticRegression(
    max_iter=2000,
    n_jobs=-1,
    class_weight=CLASS_WEIGHT,
    C=1.0,
    solver='saga',
    random_state=RANDOM_STATE,
)
metrics, report = evaluate_model(
    'LogisticRegression',
    model,
    X_train_vectors,
    y_train,
    X_valid_vectors,
    y_valid,
)
results.append(metrics)
reports['LogisticRegression'] = report
print(metrics)
print(report)


/Users/jimmydutto/Documents/GitHub/Rakuten/.venv/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


{'model': 'LogisticRegression', 'accuracy': 0.8341380122468205, 'f1_macro': 0.8211284452186229, 'f1_weighted': 0.8346963049426256}
              precision    recall  f1-score   support

          10       0.49      0.68      0.57       623
          40       0.77      0.71      0.73       502
          50       0.80      0.88      0.84       336
          60       0.85      0.87      0.86       166
        1140       0.77      0.82      0.79       534
        1160       0.95      0.95      0.95       791
        1180       0.60      0.69      0.64       153
        1280       0.78      0.57      0.66       974
        1281       0.63      0.62      0.62       414
        1300       0.87      0.95      0.91      1009
        1301       0.92      0.98      0.95       161
        1302       0.82      0.82      0.82       498
        1320       0.83      0.83      0.83       648
        1560       0.86      0.82      0.84      1015
        1920       0.90      0.93      0.91       861
    

## MultinomialNB


In [9]:
model = MultinomialNB(
    alpha=1.0,
)
metrics, report = evaluate_model(
    'MultinomialNB',
    model,
    X_train_vectors,
    y_train,
    X_valid_vectors,
    y_valid,
)
results.append(metrics)
reports['MultinomialNB'] = report
print(metrics)
print(report)


{'model': 'MultinomialNB', 'accuracy': 0.6682760244936411, 'f1_macro': 0.5841700184130787, 'f1_weighted': 0.6420236156001926}
              precision    recall  f1-score   support

          10       0.89      0.14      0.25       623
          40       0.84      0.44      0.58       502
          50       0.70      0.40      0.51       336
          60       0.99      0.69      0.82       166
        1140       0.81      0.67      0.73       534
        1160       0.92      0.90      0.91       791
        1180       0.93      0.18      0.31       153
        1280       0.45      0.62      0.52       974
        1281       0.83      0.18      0.29       414
        1300       0.72      0.95      0.82      1009
        1301       1.00      0.48      0.65       161
        1302       0.81      0.26      0.39       498
        1320       0.97      0.41      0.57       648
        1560       0.50      0.78      0.61      1015
        1920       0.87      0.84      0.86       861
        1

## Summary Table


In [10]:
results_df = pd.DataFrame(results).set_index('model')
results_df = results_df.sort_values('f1_macro', ascending=False)
results_df


,accuracy,f1_macro,f1_weighted
model,,,
LinearSVC,0.853274,0.838927,0.852486
LogisticRegression,0.834138,0.821128,0.834696
SGDClassifier,0.789390,0.780011,0.791832
MultinomialNB,0.668276,0.584170,0.642024
